In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import sys
sys.path.append('..')#tells the notebook to go up one level in the directory structure to find the src folder

from src.predictor.data import load_matches

df = load_matches()
df.head()


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A,season
0,2014-08-23,Almeria,Espanol,1,1,D,2.55,3.3,2.70,1415
1,2014-08-23,Granada,La Coruna,2,1,H,1.95,3.2,4.20,1415
2,2014-08-23,Malaga,Ath Bilbao,1,0,H,2.80,3.3,2.45,1415
3,2014-08-23,Sevilla,Valencia,1,1,D,2.05,3.5,3.50,1415
4,2014-08-24,Barcelona,Elche,3,0,H,1.10,9.0,21.00,1415


In [17]:
(df['FTR'].value_counts(normalize=True) * 100).round(1) #counts how many times a value appears 

FTR
H    45.4
A    28.6
D    26.0
Name: proportion, dtype: float64

In [18]:
df[['FTHG', 'FTAG']].mean().round(2)

FTHG    1.49
FTAG    1.13
dtype: float64

In [19]:
home = df['FTHG'].mean()
away = df['FTAG'].mean()

print(home + away) #goals per match 
print(home - away) #gap between home and away goals

2.6263157894736846
0.3622009569377991


In [20]:
df["prob_H"] = 1 / df["B365H"]
df["prob_D"] = 1 / df["B365D"]
df["prob_A"] = 1 / df["B365A"]

df["total"] = df["prob_H"] + df["prob_D"] + df["prob_A"]

df["total"].mean()

np.float64(1.0539013058126239)

In [21]:
df["pH"] = df["prob_H"] / df["total"]
df["pD"] = df["prob_D"] / df["total"]
df["pA"] = df["prob_A"] / df["total"]
#top part removes margin of error and bottom part normalizes the probabilities so they add up to 1


df[["pH", "pD", "pA"]].sum(axis=1).mean()


np.float64(1.0)

In [22]:
df["bookie_pick"] = df[["pH", "pD", "pA"]].idxmax(axis=1)
df["bookie_pick"] = df["bookie_pick"].map({"pH": "H", "pD": "D", "pA": "A"})

(df["bookie_pick"] == df["FTR"]).mean()

np.float64(0.5404306220095694)

In [23]:
df["bookie_pick"].value_counts()

bookie_pick
H    2902
A    1274
D       4
Name: count, dtype: int64

In [24]:
df[["pH", "pD", "pA"]].mean()
df["pD"].max()

np.float64(0.37532976558377934)

In [25]:
print(df[["pH", "pD", "pA"]].mean().round(3))
print(df["pD"].max())

pH    0.450
pD    0.255
pA    0.296
dtype: float64
0.37532976558377934


In [26]:
from src.predictor.features import build_features
from src.predictor.evaluate import split_by_season, log_loss, bookmaker_probs

train, valid, test = split_by_season(build_features(load_matches()))

# 1. frequency baseline: training rates, same for every match
freq = train["FTR"].value_counts(normalize=True)
freq_probs = [[freq["H"], freq["D"], freq["A"]]] * len(valid)
print("frequency:", log_loss(freq_probs, valid["FTR"]))

# 2. bookmaker
print("bookmaker:", log_loss(bookmaker_probs(valid), valid["FTR"]))

frequency: 1.0528709802374765
bookmaker: 0.9797455178546111


In [ ]:
import pandas as pd
from src.predictor.evaluate import brier_score, accuracy

results = valid["FTR"]
n = len(valid)

predictors = {
    "know nothing": [[1/3, 1/3, 1/3]] * n,
    "frequency": freq_probs,
    "bookmaker": bookmaker_probs(valid),
    "logistic regression": probs_hda,
    "lightgbm": lgbm_hda
    
}

table = pd.DataFrame(
    {
        name: {
            "log loss": log_loss(probs, results),
            "brier": brier_score(probs, results),
            "accuracy": accuracy(probs, results),
            
        }
        for name, probs in predictors.items()
    }
).T.round(3)

table

NameError: name 'probs_hda' is not defined

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.predictor.model import FEATURES, get_xy

medians = train[FEATURES].median()
X_train, y_train = get_xy(train, medians)
X_valid, y_valid = get_xy(valid, medians)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

probs = model.predict_proba(X_valid)
print(model.classes_)

['A' 'D' 'H']


In [ ]:
order = [list(model.classes_).index(c) for c in ["H", "D", "A"]]
probs_hda = probs[:, order]

print(probs_hda[:3].round(2))
print(probs_hda.sum(axis=1)[:3])    
print(probs_hda.mean(axis=0).round(3))

[[0.17 0.32 0.51]
 [0.49 0.29 0.21]
 [0.08 0.21 0.71]]
[1. 1. 1.]
[0.448 0.262 0.291]


In [ ]:
from src.predictor.model import make_lgbm

lgbm = make_lgbm()
lgbm.fit(X_train, y_train)

lgbm_probs = lgbm.predict_proba(X_valid)
order = [list(lgbm.classes_).index(c) for c in ["H", "D", "A"]]
lgbm_hda = lgbm_probs[:, order]

NameError: name 'X_train' is not defined